In [1]:
# ============================================
# Stage 3 – Dual-Token Fusion (RGB + FFT on Input)
# Full fine-tuning with DeepfakeBench metrics
# ============================================

import os, random, torch, torch.nn as nn, torch.optim as optim
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.fft as fft
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from PIL import Image
from io import BytesIO
import warnings
import torch.nn.functional as F
warnings.filterwarnings("ignore")

# ---- DeepfakeBench metric helper ----
def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer, acc


In [2]:
class FFTMagnitude(nn.Module):
    """Compute log-scaled magnitude spectrum of input images."""
    def forward(self, x):
        # x: [B, 3, H, W]
        f = torch.fft.fft2(x)
        fshift = torch.fft.fftshift(f)
        mag = torch.abs(fshift)
        mag = torch.log1p(mag)
        return mag


In [3]:
class Stage3Hybrid(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, num_layers=4):
        super().__init__()
        # ---- Backbones ----
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-2])  # keep conv part

        self.fft_layer = FFTMagnitude()
        self.proj_rgb = nn.Conv2d(2048, embed_dim, 1)
        self.proj_fft = nn.Conv2d(3, embed_dim, 1)

        # ---- Transformer ----
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4,
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # ---- Positional embeddings ----
        self.pos_embed = nn.Parameter(torch.randn(1, 200, embed_dim))

        # ---- Classifier ----
        self.cls_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 2)
        )

    

    def forward(self, x):
        # RGB branch
        rgb_feat = self.resnet(x)                        # [B,2048,10,10]
        rgb = self.proj_rgb(rgb_feat).flatten(2).transpose(1,2)   # [B,100,512]
    
        # FFT branch (on input)
        fft_img = self.fft_layer(x)                      # [B,3,320,320]
        fft_proj = self.proj_fft(fft_img)
        fft_proj = F.adaptive_avg_pool2d(fft_proj, (10,10))       # downsample to 10×10
        fft_tokens = fft_proj.flatten(2).transpose(1,2)            # [B,100,512]
    
        # Token fusion
        tokens = torch.cat([rgb, fft_tokens], dim=1)               # [B,200,512]
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]
    
        out = self.transformer(tokens)
        out = out.mean(dim=1)
        return self.cls_head(out)


In [4]:
def train_stage3():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = Stage3Hybrid().to(device)

    # ---- JPEG degradation augment ----
    def random_jpeg(img):
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(40, 90))
        buf.seek(0)
        return Image.open(buf)

    train_tfms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.1,0.05),
        transforms.RandomApply([transforms.Lambda(random_jpeg)], p=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    val_tfms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    # ---- Dataset ----
    trainset = datasets.ImageFolder(
        "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train",
        transform=train_tfms)
    valset = datasets.ImageFolder(
        "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid",
        transform=val_tfms)

    trainloader = DataLoader(trainset, batch_size=16, shuffle=True, num_workers=2)
    valloader   = DataLoader(valset, batch_size=16, shuffle=False, num_workers=2)
    real_idx = trainset.class_to_idx.get("real", 1)

    # ---- Optimiser / loss / scaler ----
    opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")
    epochs, best_auc = 10, 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for imgs, lbls in tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}", ncols=100):
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                out = model(imgs)
                loss = criterion(out, lbls)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running_loss += loss.item()

        # ---- Validation ----
        model.eval(); y_true, y_prob = [], []
        with torch.no_grad():
            for imgs, lbls in valloader:
                imgs = imgs.to(device)
                probs = torch.softmax(model(imgs), dim=1)[:, real_idx].cpu().numpy()
                y_true.extend(lbls.numpy()); y_prob.extend(probs)

        auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
        print(f"Epoch {epoch+1}: loss={running_loss/len(trainloader):.4f} | "
              f"AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")

        ckpt_path = f"/kaggle/working/stage3_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), ckpt_path)
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), "/kaggle/working/best_auc_stage3.pth")
            print(f"★ New best AUC {best_auc:.3f} (ACC={acc:.3f})")

    print("\nTraining complete!")
    print(f"Best model achieved: AUROC={best_auc:.3f}")
    return model, valloader, real_idx


In [5]:
def evaluate(model, testloader, real_idx):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.eval(); y_true, y_prob = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(testloader, desc="Testing", ncols=100):
            imgs = imgs.to(device)
            probs = torch.softmax(model(imgs), dim=1)[:, real_idx].cpu().numpy()
            y_true.extend(lbls.numpy()); y_prob.extend(probs)
    auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
    print(f"Test Results → AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")


In [6]:
if __name__ == "__main__":
    model, testloader, real_idx = train_stage3()
    evaluate(model, testloader, real_idx)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 222MB/s]
Epoch 1/10: 100%|███████████████████████████████████████████████| 6250/6250 [21:11<00:00,  4.92it/s]


Epoch 1: loss=0.1381 | AUROC=1.000 | F1=0.991 | EER=0.009 | ACC=0.991
★ New best AUC 1.000 (ACC=0.991)


Epoch 2/10: 100%|███████████████████████████████████████████████| 6250/6250 [21:05<00:00,  4.94it/s]


Epoch 2: loss=0.0600 | AUROC=1.000 | F1=0.991 | EER=0.008 | ACC=0.991


Epoch 3/10: 100%|███████████████████████████████████████████████| 6250/6250 [21:07<00:00,  4.93it/s]


...

Epoch 9: loss=0.0210 | AUROC=1.000 | F1=0.994 | EER=0.005 | ACC=0.994


Epoch 10/10: 100%|██████████████████████████████████████████████| 6250/6250 [21:06<00:00,  4.93it/s]


Epoch 10: loss=0.0189 | AUROC=1.000 | F1=0.996 | EER=0.004 | ACC=0.996
★ New best AUC 1.000 (ACC=0.996)

Training complete!
Best model achieved: AUROC=1.000


Testing: 100%|██████████████████████████████████████████████████| 1250/1250 [01:19<00:00, 15.69it/s]

Test Results → AUROC=1.000 | F1=0.996 | EER=0.004 | ACC=0.996
